In [18]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
    size_adjusted_power_comparison,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = True
_AUGMENTED_PARAM = 'x_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.10
_MC_SAMPLES = 1000
_MC_ALPHA = 0.05
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)



In [19]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83   0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [20]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [21]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [22]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [23]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: True
Augmented measurement equation: OutGap
Augmented coefficient: x_coef
Monte Carlo replications: 1000
Noise Covariance:
 [[1.208 0.    0.   ]
 [0.    1.679 0.   ]
 [0.    0.    0.078]]


In [24]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 1000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,0.898,0.525,0.041,0.009,1000,42,0.042,0.006,0.031,0.056
1,Infl,0.950,0.519,0.044,0.009,1000,53,0.053,0.007,0.041,0.069
2,Rate,1.078,0.478,0.046,0.009,1000,58,0.058,0.007,0.045,0.074


In [25]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 1000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.122,3.230,0.489,0.002,0.092,0.009,1000,69,0.069,0.008,0.055,0.086,3.0,200,4
1,cov_identity,2.864,424.159,0.000,0.012,4.862,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


In [26]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,0.360,0.020,1.272,0.278,0.482,0.006,0.041,0.002,0.003,0.032,0.009,0.0,1000,59,0.059,0.007,0.046,0.075
1,OutGap,x,-0.355,-0.089,0.269,-1.273,0.309,0.012,0.009,0.002,0.001,0.030,0.009,0.0,1000,232,0.232,0.013,0.207,0.259
2,OutGap,r,-1.544,-0.047,2.249,-0.666,0.433,0.007,0.074,0.002,0.009,0.032,0.010,0.0,1000,99,0.099,0.009,0.082,0.119
3,Infl,Pi,-0.649,-0.029,1.518,-0.410,0.462,0.006,0.049,0.002,0.003,0.033,0.009,0.0,1000,60,0.060,0.008,0.047,0.076
4,Infl,x,0.033,0.008,0.322,0.114,0.484,0.005,0.011,0.002,0.001,0.033,0.009,0.0,1000,57,0.057,0.007,0.044,0.073
5,Infl,r,-0.325,-0.006,2.688,-0.085,0.488,0.005,0.089,0.002,0.010,0.033,0.009,0.0,1000,59,0.059,0.007,0.046,0.075
6,Rate,Pi,-0.061,-0.015,0.294,-0.208,0.477,0.005,0.010,0.002,0.001,0.033,0.009,0.0,1000,56,0.056,0.007,0.043,0.072
7,Rate,x,0.032,0.034,0.062,0.482,0.458,0.006,0.002,0.002,0.000,0.033,0.010,0.0,1000,74,0.074,0.008,0.059,0.092
8,Rate,r,-0.082,-0.009,0.520,-0.131,0.486,0.005,0.017,0.002,0.002,0.033,0.009,0.0,1000,49,0.049,0.007,0.037,0.064


In [27]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-0.646,-0.053,0.846,-0.751,0.438,0.007,0.024,0.002,0.001,0.028,0.009,0.0,1000,92,0.092,0.009,0.076,0.112
1,OutGap,x,-0.246,-0.094,0.178,-1.329,0.280,0.012,0.005,0.002,0.000,0.026,0.008,0.0,1000,216,0.216,0.013,0.192,0.243
0,OutGap,r,-0.774,-0.025,2.148,-0.359,0.513,0.005,0.061,0.002,0.008,0.028,0.009,0.0,1000,43,0.043,0.006,0.032,0.057
5,Infl,Pi,-0.362,-0.024,1.011,-0.340,0.481,0.006,0.032,0.002,0.001,0.032,0.009,0.0,1000,71,0.071,0.008,0.057,0.089
4,Infl,x,-0.043,-0.011,0.214,-0.155,0.495,0.005,0.007,0.002,0.001,0.032,0.009,0.0,1000,59,0.059,0.007,0.046,0.075
3,Infl,r,0.076,0.003,2.563,0.043,0.498,0.005,0.082,0.002,0.009,0.032,0.009,0.0,1000,54,0.054,0.007,0.042,0.070
8,Rate,Pi,0.033,0.011,0.196,0.155,0.505,0.005,0.006,0.002,0.000,0.032,0.009,0.0,1000,54,0.054,0.007,0.042,0.070
7,Rate,x,0.019,0.030,0.041,0.420,0.467,0.006,0.001,0.002,0.000,0.032,0.009,0.0,1000,82,0.082,0.009,0.067,0.101
6,Rate,r,-0.104,-0.012,0.496,-0.170,0.495,0.005,0.016,0.002,0.002,0.032,0.009,0.0,1000,52,0.052,0.007,0.040,0.068


In [28]:
print("Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"]).round(3)

Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.737,-1.377,0.360,0.360,-0.0,0.0,0.0,0.028,0.020,0.041,0.041,0.0,0.0,0.0
1,OutGap,x,0.002,-0.357,-0.355,-0.355,-0.0,0.0,0.0,0.006,0.005,0.009,0.009,0.0,0.0,0.0
2,OutGap,r,-0.065,-1.479,-1.544,-1.544,-0.0,0.0,0.0,0.049,0.041,0.074,0.074,0.0,0.0,0.0
3,Infl,Pi,-0.064,-0.585,-0.649,-0.649,-0.0,0.0,0.0,0.015,0.047,0.049,0.049,0.0,0.0,0.0
4,Infl,x,0.007,0.026,0.033,0.033,-0.0,0.0,0.0,0.003,0.010,0.011,0.011,0.0,0.0,0.0
5,Infl,r,-0.071,-0.254,-0.325,-0.325,-0.0,0.0,0.0,0.027,0.088,0.089,0.089,0.0,0.0,0.0
6,Rate,Pi,0.006,-0.066,-0.061,-0.061,0.0,0.0,0.0,0.003,0.009,0.010,0.010,0.0,0.0,0.0
7,Rate,x,-0.000,0.032,0.032,0.032,0.0,0.0,0.0,0.001,0.002,0.002,0.002,0.0,0.0,0.0
8,Rate,r,-0.022,-0.060,-0.082,-0.082,0.0,0.0,0.0,0.006,0.017,0.017,0.017,0.0,0.0,0.0


In [29]:
print("Innovation decomposition on raw predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"]).round(3)

Innovation decomposition on raw predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.815,-2.461,-0.646,-0.646,-0.0,0.0,0.0,0.018,0.010,0.024,0.024,0.0,0.0,0.0
1,OutGap,x,0.285,-0.531,-0.246,-0.246,0.0,0.0,0.0,0.004,0.003,0.005,0.005,0.0,0.0,0.0
2,OutGap,r,-1.036,0.261,-0.774,-0.774,-0.0,0.0,0.0,0.057,0.034,0.061,0.061,0.0,0.0,0.0
3,Infl,Pi,-0.024,-0.338,-0.362,-0.362,-0.0,0.0,0.0,0.010,0.031,0.032,0.032,0.0,0.0,0.0
4,Infl,x,-0.003,-0.040,-0.043,-0.043,-0.0,0.0,0.0,0.002,0.007,0.007,0.007,0.0,0.0,0.0
5,Infl,r,-0.038,0.114,0.076,0.076,0.0,0.0,0.0,0.026,0.082,0.082,0.082,0.0,0.0,0.0
6,Rate,Pi,0.007,0.026,0.033,0.033,0.0,0.0,0.0,0.002,0.006,0.006,0.006,0.0,0.0,0.0
7,Rate,x,0.001,0.018,0.019,0.019,0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
8,Rate,r,-0.019,-0.085,-0.104,-0.104,0.0,0.0,0.0,0.006,0.016,0.016,0.016,0.0,0.0,0.0


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.


In [30]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()



## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [31]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,1.306,-1293.991,-1125.742,336.497,0.0,0.003,1.244,0.508,1.763,0.0,1000,1000,1.0,0.0,0.996,1.0


In [32]:
res_mle

OptimizationResult(kind='mle', x=array([1.16703589]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(1.1670358902592632), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(1103.8767212013388), loglik=np.float64(-1103.8767212013388), logprior=np.float64(0.0), logpost=np.float64(-1103.8767212013388), nfev=14, nit=6, raw=  message: CONVERGENCE

## Serial Autocorrelation Tests for the Augmented Model

In [33]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.993,0.356,0.076,0.009,1000,163,0.163,0.012,0.141,0.187
1,Infl,3.869,0.181,0.106,0.007,1000,407,0.407,0.016,0.377,0.438
2,Rate,1.055,0.484,0.046,0.009,1000,56,0.056,0.007,0.043,0.072


In [34]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.122,3.230,0.489,0.002,0.092,0.009,1000,69,0.069,0.008,0.055,0.086,3.0,200,4
1,cov_identity,2.864,424.159,0.000,0.012,4.862,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.098,2.926,0.515,0.001,0.079,0.009,1000,52,0.052,0.007,0.04,0.068,3.0,200,4
1,cov_identity,0.466,74.945,0.001,0.002,1.229,0.000,1000,996,0.996,0.002,0.99,0.998,6.0,200,4


Reference-minus-augmented moment distance comparison:


,test,n_replications,distance_ref,mc_se_distance_ref,distance_aug,mc_se_distance_aug,distance_improvement,mc_se_distance_improvement,stat_ref,mc_se_stat_ref,stat_aug,mc_se_stat_aug,stat_improvement,mc_se_stat_improvement,aug_closer_rate,aug_closer_rate_mc_se,aug_closer_ci_low,aug_closer_ci_high
0,mean_zero_hac,1000,0.122,0.002,0.098,0.001,0.024,0.001,3.230,0.092,2.926,0.079,0.304,0.026,0.866,0.011,0.843,0.886
1,cov_identity,1000,2.864,0.012,0.466,0.002,2.398,0.012,424.159,4.862,74.945,1.229,349.214,4.066,1.000,0.000,0.996,1.000
